# Qwen3-TTS on Google Colab

Run Qwen3-TTS with CUDA GPU acceleration on Google Colab.

**Requirements:** A Colab runtime with GPU (T4 or better).

## Setup

In [ ]:
# Install system dependencies
!apt-get update && apt-get install -y ffmpeg

# Clone the repo (or pull latest if already cloned)
!git clone https://github.com/eepstein201/Qwen3-TTS-Advanced-EME.git ~/Qwen3-TTS_UserFiles 2>/dev/null || (cd ~/Qwen3-TTS_UserFiles && git pull)

# Install Python dependencies
!pip install -q -r ~/Qwen3-TTS_UserFiles/requirements-cuda.txt

print('Setup complete!')

In [ ]:
# Configure for CUDA backend
import json, os

config_path = os.path.expanduser('~/Qwen3-TTS_UserFiles/config.json')
with open(config_path) as f:
    config = json.load(f)

config['advanced']['backend'] = 'torch'
config['advanced']['dtype'] = 'float16'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Start the TTS server in the background
import subprocess, time

server = subprocess.Popen(
    ['python', os.path.expanduser('~/Qwen3-TTS_UserFiles/voice_server.py')],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)  # Wait for server startup
print(f'Server started (PID: {server.pid})')

In [ ]:
# Launch Gradio UI with public URL
import sys
sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))

from voice_ui import build_ui
demo = build_ui()
demo.launch(server_name='0.0.0.0', share=True)

In [ ]:
# Quick generation example (without UI)
from voice_client import TTSClient

client = TTSClient()
output = client.generate(
    'Hello from Google Colab! This is Qwen3 TTS running on a GPU.',
    mode='design',
    description='A warm, friendly voice with clear articulation',
    output='colab_test.wav'
)
print(f'Generated: {output}')

# Play in notebook
from IPython.display import Audio
Audio(output)